# Оценка качества моделей классификации.

Будем тренироваться интерпретировать качество моделей классификации.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

%pylab inline

### Задача: датасет о кредитном скоринге. 
* Колонки 0, 1, ..., 13 - анонимизированные характеристики клиентов (некоторые числовые, некоторые категориальные). 
* Target - целевая переменная: 1 - выдать кредит, 0 - не выдать кредит.

На этих данных было обучено три различных алгоритма классификации и получены предсказания:
* pred1_probs - предсказанные вероятности положительного класса, полученные алгоритмом 1.
* pred1_classes - предсказанные алгоритмом 1 классы
* pred2_probs, pred2_classes, pred3_probs, pred3_classes - аналогичные величины для алгоритмов 2 и 3

Наша задача - оценить качество каждого из трех алгоритмов и разобраться, какой из алгоритмов лучше всего работает в данной задаче.

In [ ]:
df = pd.read_csv("credit_scoring_example1.csv")
df.head()

In [ ]:
df.target.unique()

In [ ]:
df.shape

Посмотрим, сбалансированная ли выборка.

In [ ]:
len(df[df.target == 0]), len(df[df.target == 1])

Оценим **accuracy** - долю правильных ответов каждого из алгоритмов.

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
help(accuracy_score)

In [ ]:
print('алгоритм 1:', accuracy_score(df['target'], df['pred1_classes']))
print('алгоритм 2:', accuracy_score(df['target'], df['pred2_classes']))
print('алгоритм 3:', accuracy_score(df['target'], df['pred3_classes']))

***С точки зрения метрики accuracy второй алгоритм работает немного лучше остальных***. Посмотрим на другие метрики.

Теперь посмотрим на **precision** и **recall**.

In [ ]:
from sklearn.metrics import precision_score, recall_score

In [ ]:
help(precision_score)

In [ ]:
help(recall_score)

In [ ]:
print('алгоритм 1:')
print('точность:', precision_score(df['target'], df['pred1_classes']))
print('полнота:',  recall_score(df['target'], df['pred1_classes']))
print('алгоритм 2:')
print('точность:', precision_score(df['target'], df['pred2_classes']))
print('полнота:',  recall_score(df['target'], df['pred2_classes']))
print('алгоритм 3:')
print('точность:', precision_score(df['target'], df['pred3_classes']))
print('полнота:',  recall_score(df['target'], df['pred3_classes']))

Самым полезным инструментом в практических задачах является ***матрица ошибок***. Смотря на неё и регулируя порог, определяющий классы, мы можем достичь оптимального решения нашей задачи.

In [ ]:
import itertools
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(cm, classes,
                          normalize = False,
                          title = 'Confusion matrix',
                          cmap = plt.cm.Blues):
 
    plt.imshow(cm, interpolation = 'nearest', cmap = cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)
    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
print('алгоритм 1')
plot_confusion_matrix(confusion_matrix(df['target'], df['pred1_classes']), classes=['0','1'],
                        title='Confusion matrix, without normalization')
show()
print('алгоритм 2')
plot_confusion_matrix(confusion_matrix(df['target'], df['pred2_classes']), classes=['0','1'],
                        title='Confusion matrix, without normalization')
show()
print('алгоритм 3')
plot_confusion_matrix(confusion_matrix(df['target'], df['pred3_classes']), classes=['0','1'],
                        title='Confusion matrix, without normalization')
show()

Давайте определимся с целями нашего предсказания:
    
* **Вариант 1**: хотим выдать кредит максимальному числу людей среди тех, кто мог бы его вернуть. При этом не хотим много ошибаться (общее число ошибок не более 35%).
  
1) Мы не хотим, чтобы алгоритм много ошибался: accuracy $\geq$ 0.65.

2) Кроме того полнота должна быть как можно больше (число в левом нижнем квадрате матрицы ошибок минимально).

Будем изменять порог вероятности в цикле и смотреть на результаты.

Смотрим на **модель 1**:

In [ ]:
df['pred1_probs'] > 0.2

In [ ]:
for t in np.arange(0.1, 1, 0.1):
    pred1_thr = [1 if x >= t else 0 for x in df['pred1_probs']]
    print('threshold =', t)
    print('полнота(recall):', recall_score(df['target'],pred1_thr))
    print('accuracy:', accuracy_score(df['target'],pred1_thr))
    print()

In [ ]:
for t in np.arange(0.35, 0.6, 0.025):
    pred1_thr = [1 if x >= t else 0 for x in df['pred1_probs']]
    print('threshold =', t)
    print('полнота(recall):', recall_score(df['target'],pred1_thr))
    print('accuracy:', accuracy_score(df['target'],pred1_thr))
    print()

***Наилучший результат:***

threshold = 0.375

полнота(recall): 0.71

accuracy: 0.65

In [ ]:
pred1_thr = [1 if x >= 0.375 else 0 for x in df['pred1_probs']]

plot_confusion_matrix(confusion_matrix(df['target'], pred1_thr), classes=['0','1'],
                        title='Confusion matrix, without normalization')

Теперь посмотрим на **модель 2**:

In [ ]:
for t in np.arange(0.1,1,0.1):
    pred2_thr = [1 if x >= t else 0 for x in df['pred2_probs']]
    print('threshold =', t)
    print('полнота(recall):', recall_score(df['target'],pred2_thr))
    print('accuracy:', accuracy_score(df['target'],pred2_thr))
    print()

In [ ]:
for t in np.arange(0.25,0.5,0.025):
    pred2_thr = [1 if x >= t else 0 for x in df['pred2_probs']]
    print('threshold =', t)
    print('полнота(recall):', recall_score(df['target'],pred2_thr))
    print('accuracy:', accuracy_score(df['target'],pred2_thr))
    print()

***Наилучший результат:***

threshold = 0.3

полнота(recall): 0.837
    
accuracy: 0.675

In [ ]:
pred2_thr = [1 if x >= 0.3 else 0 for x in df['pred2_probs']]

plot_confusion_matrix(confusion_matrix(df['target'], pred2_thr), classes=['0','1'],
                        title='Confusion matrix, without normalization')

Посмотрим на **модель 3**:

In [ ]:
for t in np.arange(0.1,1,0.1):
    pred3_thr = [1 if x >= t else 0 for x in df['pred3_probs']]
    print('threshold =', t)
    print('полнота(recall):', recall_score(df['target'],pred3_thr))
    print('accuracy:', accuracy_score(df['target'],pred3_thr))
    print()

In [ ]:
for t in np.arange(0.35,0.6,0.025):
    pred3_thr = [1 if x >= t else 0 for x in df['pred3_probs']]
    print('threshold =', t)
    print('полнота(recall):', recall_score(df['target'],pred3_thr))
    print('accuracy:', accuracy_score(df['target'],pred3_thr))
    print()

***Наилучший результат:***

threshold = 0.4

полнота(recall): 0.414
    
accuracy: 0.646 (наибольшая доля правильных ответов)

***Модель не проходит по критериям (доля правильных ответов меньше 0.65)!***

In [ ]:
pred3_thr = [1 if x >= 0.4 else 0 for x in df['pred3_probs']]

plot_confusion_matrix(confusion_matrix(df['target'], pred3_thr), classes=['0','1'],
                        title='Confusion matrix, without normalization')

**Выводы:** с поставленной задачей лучше всего справляется модель 2 (полнота: 0.837,
accuracy: 0.675). 

Наша цель может звучать и по-другому:
    
* **Вариант 2**: среди тех, кому модель выдает кредит, должно быть меньше всего ошибок (мало людей, кто его на самом деле не вернет - на них будем терять деньги). 
    При этом не хотим много ошибаться (общее число ошибок не более 35%).
  
1) Мы не хотим, чтобы алгоритм много ошибался: accuracy $\geq$ 0.65.

2) Кроме того точность должна быть как можно больше (число в правом верхнем квадрате матрицы ошибок минимально).

Будем изменять порог вероятности в цикле и смотреть на результаты.

Смотрим на **модель 1**:

In [ ]:
pred1_thr = [1 if x >= 0.55
             else 0 for x in df['pred1_probs']]

plot_confusion_matrix(confusion_matrix(df['target'], pred1_thr), classes=['0','1'],
                        title='Confusion matrix, without normalization')

In [ ]:
precision_score(df['target'], pred1_thr)

In [ ]:
accuracy_score(df['target'], pred1_thr)

Смотрим на **модель 2**:

In [ ]:
for t in np.arange(0.25,0.8,0.025):
    pred2_thr = [1 if x >= t else 0 for x in df['pred2_probs']]
    print('threshold =', t)
    print('точность(precision):', precision_score(df['target'],pred2_thr))
    print('accuracy:', accuracy_score(df['target'],pred2_thr))
    print()

***Наилучший результат:***

threshold = 0.725

точность(precision): 0.837
    
accuracy: 0.65

In [ ]:
pred2_thr = [1 if x >= 0.725 else 0 for x in df['pred2_probs']]

plot_confusion_matrix(confusion_matrix(df['target'], pred2_thr), classes=['0','1'],
                        title='Confusion matrix, without normalization')

In [ ]:
precision_score(df['target'], pred2_thr)

In [ ]:
accuracy_score(df['target'], pred2_thr)

Смотрим на **модель 3**:

In [ ]:
for t in np.arange(0.1,1,0.1):
    pred3_thr = [1 if x >= t else 0 for x in df['pred3_probs']]
    print('threshold =', t)
    print('точность(precision):', precision_score(df['target'],pred3_thr))
    print('accuracy:', accuracy_score(df['target'],pred3_thr))
    print()

In [ ]:
for t in np.arange(0.4,0.65,0.025):
    pred3_thr = [1 if x >= t else 0 for x in df['pred3_probs']]
    print('threshold =', t)
    print('точность(precision):', precision_score(df['target'],pred3_thr))
    print('accuracy:', accuracy_score(df['target'],pred3_thr))
    print()

***Наилучший результат:***

threshold = 0.575

точность(precision): 0.714
    
accuracy: 0.642
    
***Модель не проходит по критериям (точность меньше 0.65)!***

In [ ]:
pred3_thr = [1 if x >= 0.575 else 0 for x in df['pred3_probs']]

plot_confusion_matrix(confusion_matrix(df['target'], pred3_thr), classes=['0','1'],
                        title='Confusion matrix, without normalization')

Также (**Вариант 3**) можно потребовать, чтобы и точность, и полнота были высокими, то есть обе ошибки в матрице ошибок были маленькие. Как вариант - можно максимизировать F1-score (среднее гармоническое между точностью и полнотой).

Найдем порог для достижения максимального F1-score для наилучшей модели (вторая модель) и построим матрицу ошибок (***проверьте, что F1-score у двух оставшихся моделей ниже!***).

In [ ]:
from sklearn.metrics import f1_score

for t in np.arange(0.1,1,0.1):
    pred1_thr = [1 if x >= t else 0 for x in df['pred1_probs']]
    print('threshold =', t)
    print('f1-score:', f1_score(df['target'],pred1_thr))
    print('accuracy:', accuracy_score(df['target'],pred1_thr))
    print()

In [ ]:
from sklearn.metrics import f1_score

for t in np.arange(0.35,0.6,0.025):
    pred1_thr = [1 if x >= t else 0 for x in df['pred1_probs']]
    print('threshold =', t)
    print('f1-score:', f1_score(df['target'],pred1_thr))
    print('accuracy:', accuracy_score(df['target'],pred1_thr))
    print()

***Наилучший результат:***

threshold = 0.375

f1-score: 0.644
    
accuracy: 0.65

In [ ]:
pred1_thr = [1 if x >= 0.375 else 0 for x in df['pred1_probs']]

plot_confusion_matrix(confusion_matrix(df['target'], pred1_thr), classes=['0','1'],
                        title='Confusion matrix, without normalization')

Посмотрим на roc-auc.

In [ ]:
from sklearn.metrics import roc_auc_score

print('алгоритм 1:', roc_auc_score(df['target'], df['pred1_probs']))
print('алгоритм 2:', roc_auc_score(df['target'], df['pred2_probs']))
print('алгоритм 3:', roc_auc_score(df['target'], df['pred3_probs']))

Построим roc-кривую для каждой модели и отметим на графике roc-auc.

In [ ]:
from sklearn.metrics import roc_curve, auc

In [ ]:
help(roc_curve)

In [ ]:
help(auc)

In [ ]:
for n in range(3):
    print('Модель ' + str(n + 1) + ':')
    fpr, tpr, threshold = roc_curve(df['target'], df['pred' + str(n+1) + '_probs'])
    roc_auc = auc(fpr, tpr)
    plt.title('Receiver Operating Characteristic')
    plt.plot(fpr, tpr, 'b', label = 'AUC = %0.2f' % roc_auc)
    plt.legend(loc = 'lower right')
    plt.plot([0, 1], [0, 1],'r--')
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.ylabel('True Positive Rate')
    plt.xlabel('False Positive Rate')
    plt.show()

### Выводы

**1. В данной задаче модель 2 выигрывает по всем метрикам. Однако не во всех задачах одна и та же модель будет лучшей относительно всех метрик!** 

**2. Качество решения задачи нельзя улучшить с помощью выбора метрики или настройки порога, но с помощью настройки порога можно добиться наилучшего результата в рамках поставленной задачи. Чтобы улучшить качество решения - необходимо изменять саму модель.**

#### Что ещё можно сделать

В python есть удобная функция classification_report, которая выводит значения нескольких метрик.

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(df['target'], df['pred1_classes']))

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(df['target'], df['pred2_classes']))

In [ ]:
df['pred2_classes'].unique()

In [ ]:
df.head()

In [ ]:
help(classification_report)

### Bias-Variance decomposition

Вспомним, что функцию потерь в задачах регрессии или классификации можно разложить на три компоненты: смещение (bias), дисперсию (variance) и шум (noise). Эти компоненты позволяют описать сложность алгоритма, альтернативно сравнению ошибок на тренировочной и тестовой выборках. Хотя такое разложение можно построить для произвольной функции потерь, наиболее просто (и классически) оно строится для среднеквадратичной функции в задаче регрессии, что мы и рассмотрим ниже. 

Пусть $(X, y)$ – некоторая выборка. Обучим интересующий нас алгоритм на этой выборке и сделаем предсказания на ней. Обозначим предсказания как $\hat{y}$. Тогда 

$$
\mathrm{bias} := \mathbb{E}(\hat{y}) - y.
$$

$$
\mathrm{variance} := \mathbb{E}[\mathbb{E}(\hat{y}) - \hat{y}]^2
$$

$$
\mathrm{noise} := \mathbb{E}[y - \mathbb{E}(y)]^2
$$

Ожидаемую среднеквадратичную ошибку для различных вариаций тренировочной выборки можно разложить как

$$
\mathrm{E}[y - \hat{y}]^2 = \mathrm{bias}^2 + \mathrm{variance} + \mathrm{noise}.
$$

**Задание для самых смелых:** покажите, что это разложение корректно. Проверьте себя [здесь](https://github.com/esokolov/ml-course-hse/blob/master/2020-fall/lecture-notes/lecture08-ensembles.pdf).

Разберёмся с интерпретацией компонент. 

- $\mathrm{Bias}$ – показывает отклонение среднего ответа алгоритма от ответа идеального алгоритма. $\mathrm{Bias}$ отражает ошибку модели, возникающую из-за простоты модели. Высокое смещение обычно является показателем того, что модель недообучена.


- $\mathrm{Variance}$ – показывает разброс ответов алгоритмов относительно среднего ответа алгоритма. Показывает, насколько сильно небольшие изменения в обучающей выборке скажутся на предсказаниях алгоритма. $\mathrm{Variance}$ отражает ошибку модели, возникающую из-за чрезмерной сложности модели. Высокая дисперсия обычно является показателем того, что модель переобучена.


- $\mathrm{Noise}$ – ошибка идеального классификатора, естественный неустранимый шум в данных. 

Посмотрим наглядно на примере полиномиальной регрессии.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Выборка
N = 10
X = np.linspace(-5, 5, N).reshape(-1, 1)
y = np.sin(X) + np.random.normal(0, 0.2, size = N).reshape(-1, 1)

In [ ]:
X

In [ ]:
y

In [ ]:
help(np.ones_like)

In [ ]:
np.ones_like(y)

In [ ]:
np.mean(y) * np.ones_like(y)

In [ ]:
# Очень простая модель (регрессия на константу)
too_simple_model_predictions = np.mean(y) * np.ones_like(y)

# В меру сложная модель
X_ok = np.hstack([X, X ** 2, X ** 3])
ok_model = LinearRegression()
ok_model.fit(X_ok, y)
ok_model_predictions = ok_model.predict(X_ok)

# Очень сложная модель
X_compl = np.hstack([X, X ** 2, X ** 3, X ** 4, X ** 5, X ** 6, X ** 7, X ** 8, X ** 9, X ** 10])
compl_model = LinearRegression()
compl_model.fit(X_compl, y)
compl_model_predictions = compl_model.predict(X_compl)

plt.figure(figsize = (10, 7))

plt.scatter(X, y)
plt.plot(X, too_simple_model_predictions, label = 'Очень простая модель')
plt.plot(X, ok_model_predictions, label = 'В меру сложная модель')
plt.plot(X, compl_model_predictions, label = 'Очень сложная модель')
plt.grid()
plt.legend();

In [ ]:
plt.scatter(X, y)
plt.show()

- Очень простая модель имеет большое смещение (bias), но малую (нулевую) дисперсию (variance). Модель явно недообучена.

- В меру сложная модель имеет небольшое смещение (bias) и небольшую дисперсию (variance).

- Очень сложная модель имеет небольшое смещение (bias), но большую дисперсию (variance). Модель явно переобучена.

**Задание:** пользуясь определениями выше, объясните, почему это так.

**Задание:** прокомментируйте величину смещения и дисперсии для следующих моделей:

1. Линейная регрессия, обучаемая на большой выборке без выбросов и линейно зависимых признаков. (малые смещение и дисперсия)
2. Решающее дерево, которое строится до тех пор, пока в листах не окажется по одному объекту. (малое смещение, высокая дисперсия)
3. Логистическая регрессия, относящая все точки к одному классу. (большое смещение, малая дисперсия) 

### 1.A. Bias-Variance tradeoff

Из описания выше можно заметить, что при обучении моделей возникает выбор между смещением и дисперсией: недообученная модель имеет низкую дисперсию, но высокое смещение, а переобученная – низкое смещение, но высокую дисперсию. Этот выбор можно отобразить на картинке ([источник](https://www.bradyneal.com/bias-variance-tradeoff-textbooks-update)).

![](https://www.bradyneal.com/img/bias-variance/fortmann-roe-bias-variance.png)

Вывод из неё очевиден: строить следует оптимальные по сложности модели. 

Возникает ли такой выбор при обучении любой модели? Последние исследования показывают, что поведение ошибки при обучении некоторых (современных) моделей не соответствует такой U-образной форме. Например, было показано, что ошибка продолжает убывать при расширении (увеличении числа слоёв) нейронных сетей:

<img src="https://www.bradyneal.com/img/bias-variance/neyshabur.jpg" alt="drawing" width="400"/>

В таких моделях поведение ошибки приобретает сложный вид:


<img src="https://www.bradyneal.com/img/bias-variance/double_descent.jpg" alt="drawing" width="800"/>

### 3. От деревьев к случайному лесу

#### 3.1 Решающее дерево

Мотивацию построения алгоритма случайного леса (Random Forest) удобно рассматривать в терминах смещения и дисперсии. Начнём с построения решающего дерева.

In [ ]:
#!pip install --upgrade mlxtend

In [ ]:
# Пример отсюда: http://rasbt.github.io/mlxtend/user_guide/evaluate/bias_variance_decomp/

from sklearn.model_selection import train_test_split
from mlxtend.data import boston_housing_data

X, y = boston_housing_data()
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.3,
                                                    random_state = 123,
                                                    shuffle = True)

In [ ]:
X

In [ ]:
y

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

In [ ]:
help(DecisionTreeRegressor)

In [ ]:
# TODO: обучите решающее дерево без ограничений на тренировочной выборке
# TODO: рассчитайте MSE на тренировочной и тестовой выборках

dt = DecisionTreeRegressor(random_state = 123)
dt.fit(X_train, y_train)
print("MSE_train: ", mean_squared_error(y_train, dt.predict(X_train)))
print("MSE_test: ", mean_squared_error(y_test, dt.predict(X_test)))

In [ ]:
DecisionTreeRegressor(random_state = 123).fit(X_train, y_train)

In [ ]:
from mlxtend.evaluate import bias_variance_decomp

# TODO: воспользуйтесь функцией bias_variance_decomp и выведите среднее смещение и среднюю дисперсию модели
# на тестовой выборке
_, avg_bias, avg_var = bias_variance_decomp(dt, X_train, y_train, X_test, y_test, 
                                            loss = 'mse', random_seed = 123)

In [ ]:
help(bias_variance_decomp)

In [ ]:
avg_bias

In [ ]:
avg_var